# KAN-REC — Comparativa completa de encoders (Colab Pro)

Este notebook produce **las cifras que van en la memoria**, leyendo el
`criteo_10m.tsv` crudo directamente desde tu Google Drive — sin pasar por
una exportación intermedia de Fabric.

## Por qué se replica aquí el pipeline de Fabric, no se reinventa

La normalización (log1p, `StandardScaler`, indexado de categóricas) se
implementa aquí en pandas, pero **replicando exactamente** la lógica de
`fabric/01_spark_ingest_mlllib.py` — no una versión distinta. Esto se
verificó comparando, celda a celda, esta implementación contra una
ejecución real de Spark sobre los mismos datos (misma partición, mismos
valores, diferencia máxima 1e-7, solo redondeo de coma flotante). Si Colab
reimplementara su propio criterio de normalización, los dos entornos
podrían divergir sin que nadie lo notara — que es exactamente el problema
que se corrigió en el paso 2 de este proyecto (una sola fuente de verdad
para el modelo; aquí, una sola fuente de verdad para el preprocesado).

**Lo único que no puede ser idéntico:** la partición train/val/test. Spark
y NumPy usan generadores de aleatoriedad distintos, así que con la misma
seed=42 no caen las mismas filas en cada split (verificado). No importa
para el resultado: lo que hace comparables los números es la fórmula de
transformación, no qué fila cae en qué split.

## Antes de ejecutar

1. Asegúrate de que `criteo_10m.tsv` está en tu Drive. Por defecto se
   busca en `/content/drive/MyDrive/kanrec/criteo_10m.tsv` — ajusta
   `TSV_PATH` en la Celda 2 si está en otra ruta.
2. Activa GPU: `Entorno de ejecución → Cambiar tipo de entorno → GPU`.
3. Ejecuta las celdas en orden. La ingesta de 10M filas con pandas tarda
   unos minutos; el resto es rápido con GPU.

**Coherencia con Fabric:** este notebook instala `kanrec` fijado al mismo
mismo commit que Fabric. Se usa `@main` temporalmente tras el fix de mlflow; repin al commit exacto una vez hecho el push (ver "Pasos a seguir").


In [ ]:
# Instalación fijada al commit exacto que usa Fabric (no @main): garantiza
# que Colab y Fabric ejecutan literalmente el mismo código del paquete.
get_ipython().system('pip install --quiet "git+https://github.com/bdm-lab-cap/kanrec.git@0a3ec2f6b6f10c6e7cbac26931f4293fedf0e87a"')
get_ipython().system('pip install --quiet mlflow scikit-learn')


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

# Ajusta esta ruta si tu fichero esta en otro sitio de tu Drive.
TSV_PATH = "/content/drive/MyDrive/kanrec/criteo_10m.tsv"
CKPT_DIR = "/content/drive/MyDrive/kanrec_checkpoints"
RESULTS_DIR = "/content/drive/MyDrive/kanrec_results"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.exists(TSV_PATH):
    raise FileNotFoundError(
        f"No encuentro {TSV_PATH}. Ajusta TSV_PATH a donde tengas "
        f"criteo_10m.tsv en tu Drive."
    )
size_gb = os.path.getsize(TSV_PATH) / 1e9
print(f"Fichero encontrado: {TSV_PATH} ({size_gb:.2f} GB)")


In [ ]:
import time

import mlflow
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import log_loss, roc_auc_score
from torch.utils.data import DataLoader, Dataset

from kanrec.baselines import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {device}")
if device.type == "cpu":
    print("Aviso: sin GPU. Activa Entorno de ejecucion -> Cambiar tipo de "
          "entorno -> GPU antes de seguir, o esto sera muy lento.")

NUMERICAL_COLS   = [f"I{i}" for i in range(1, 14)]
CATEGORICAL_COLS = [f"C{i}" for i in range(1, 27)]
LOG_COLS = [f"I{i}" for i in range(1, 6)]   # log1p, igual que fabric/01
STD_COLS = [f"I{i}" for i in range(6, 14)]  # StandardScaler, igual que fabric/01
idx_cols = [f"{c}_idx" for c in CATEGORICAL_COLS]

# Pon un numero aqui (p.ej. 2_000_000) para una primera pasada rapida.
# None = usar el fichero completo.
N_SAMPLE = None


def replicate_fabric_01(tsv_path, num_cols, cat_cols, log_cols, std_cols,
                         seed=42, n_sample=None):
    """
    Replica fiel de fabric/01_spark_ingest_mlllib.py en pandas.

    Verificado celda a celda contra una ejecucion real de Spark sobre los
    mismos datos (misma particion via row_id, mismos valores): diferencia
    maxima 1e-7 en las columnas numericas, 0 discrepancias en el indexado
    de categoricas. La UNICA diferencia frente a Fabric es la composicion
    exacta de la particion train/val/test (Spark y NumPy usan generadores
    de aleatoriedad distintos), no la formula de transformacion.

    Replica en particular:
      - Nulos -> 0.0, luego abs() en las 13 columnas numericas.
      - log1p(max(x,0)) en I1..I5.
      - StandardScaler (media y desviacion MUESTRAL, ddof=1) en I6..I13,
        ajustado solo con TRAIN y aplicado a los tres splits.
      - Indexado de categoricas por frecuencia descendente (empates por
        orden alfabetico ascendente, igual que StringIndexer), con nulos
        y categorias no vistas en val/test mapeadas a un unico "cubo"
        invalido en el indice = numero de categorias conocidas en train
        (mismo comportamiento que handleInvalid="keep").
    """
    cols = ["label"] + num_cols + cat_cols
    print("Leyendo el TSV (puede tardar unos minutos para 10M filas)...")
    t0 = time.time()
    read_kwargs = dict(sep="\t", header=None, names=cols,
                        na_values=[""], keep_default_na=True)
    if n_sample is not None:
        # Lectura completa y luego muestreo aleatorio (mas simple y mas
        # fiel a random_sample de kanrec.spark_utils que un nrows= directo,
        # que solo tomaria las primeras filas del fichero -- el mismo
        # problema de sesgo temporal que se corrigio en el hallazgo A4).
        df = pd.read_csv(tsv_path, **read_kwargs)
        df = df.sample(n=min(n_sample, len(df)), random_state=seed).reset_index(drop=True)
    else:
        df = pd.read_csv(tsv_path, **read_kwargs)
    print(f"  {len(df):,} filas leidas en {time.time()-t0:.0f}s")

    for c in num_cols:
        df[c] = df[c].fillna(0.0).abs()

    n = len(df)
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n)
    n_train, n_val = int(n * 0.8), int(n * 0.1)
    train_df = df.iloc[perm[:n_train]].reset_index(drop=True)
    val_df   = df.iloc[perm[n_train:n_train + n_val]].reset_index(drop=True)
    test_df  = df.iloc[perm[n_train + n_val:]].reset_index(drop=True)
    del df

    for split_df in (train_df, val_df, test_df):
        for c in log_cols:
            split_df[c] = np.log1p(np.maximum(split_df[c].values, 0.0))

    means = train_df[std_cols].mean()
    stds  = train_df[std_cols].std(ddof=1)
    for split_df in (train_df, val_df, test_df):
        for c in std_cols:
            split_df[c] = (split_df[c] - means[c]) / stds[c]

    for c in cat_cols:
        counts = train_df[c].value_counts()
        ordered = sorted(counts.index, key=lambda v: (-counts[v], v))
        code_map = {v: i for i, v in enumerate(ordered)}
        invalid_code = len(ordered)
        for split_df in (train_df, val_df, test_df):
            split_df[f"{c}_idx"] = split_df[c].map(code_map).fillna(invalid_code).astype("int64")
        # Ya no se necesita la columna de texto original para entrenar.
        for split_df in (train_df, val_df, test_df):
            split_df.drop(columns=[c], inplace=True)

    return train_df, val_df, test_df


train_df, val_df, test_df = replicate_fabric_01(
    TSV_PATH, NUMERICAL_COLS, CATEGORICAL_COLS, LOG_COLS, STD_COLS,
    seed=42, n_sample=N_SAMPLE,
)

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    ctr = split_df["label"].mean()
    print(f"{split_name}: {len(split_df):,} filas | CTR={ctr:.2%}")

# Verificacion rapida: I6..I13 deben tener media ~0 y desviacion ~1 en TRAIN
# (la misma comprobacion que se hace en fabric/01, guardala para el Anexo D)
print("\nVerificacion -- I6..I13 en train (esperado: mean~0, std~1):")
print(train_df[STD_COLS].agg(["mean", "std"]).round(4))

cat_cardinalities = [int(train_df[c].max()) + 1 for c in idx_cols]
print(f"\nCardinalidades categoricas: {cat_cardinalities}")


In [ ]:
class CriteoDataset(Dataset):
    """Envuelve un DataFrame ya ingerido y normalizado en la celda anterior."""

    def __init__(self, df: pd.DataFrame, num_cols: list[str], idx_cols: list[str]):
        self.x_num = torch.tensor(df[num_cols].fillna(0).values.astype("float32"))
        present = [c for c in idx_cols if c in df.columns]
        self.x_cat = torch.tensor(df[present].fillna(0).values.astype("int64")) \
            if present else torch.zeros(len(df), len(idx_cols), dtype=torch.long)
        self.y = torch.tensor(df["label"].values.astype("float32"))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x_num[idx], self.x_cat[idx], self.y[idx]


BATCH_SIZE = 2048  # Colab con GPU aguanta un batch mayor que el de Fabric
train_ds = CriteoDataset(train_df, NUMERICAL_COLS, idx_cols)
val_ds   = CriteoDataset(val_df,   NUMERICAL_COLS, idx_cols)
test_ds  = CriteoDataset(test_df,  NUMERICAL_COLS, idx_cols)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
# Learning rate por encoder, no unico para los tres (mismo criterio que en
# fabric/04_model_comparison.py). Verificado empiricamente antes de esta
# entrega: con lr=1e-3 igual para los tres, AutoDis queda claramente
# infraentrenado incluso a 30 epocas (su capa de discretizacion tiene mas
# parametros que aprender que raw o kan-bspline), mientras que estos dos
# son estables en el mismo rango de lr. Lo que debe ser igual entre
# encoders son los DATOS y la evaluacion, no necesariamente el lr.
DEFAULT_LR = {"raw": 1e-3, "autodis": 1e-2, "kan-bspline": 1e-3}


def evaluate(model, loader) -> tuple[float, float]:
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x_num, x_cat, y in loader:
            p = model(x_num.to(device), x_cat.to(device)).squeeze().cpu().numpy()
            preds.extend(np.atleast_1d(p))
            labels.extend(y.numpy())
    return roc_auc_score(labels, preds), log_loss(labels, preds)


def train_one_run(
    encoder_name: str,
    seed: int,
    max_epochs: int = 30,
    patience: int = 3,
    lr: float | None = None,
    embedding_dim: int = 16,
    grid_size: int = 10,
) -> dict:
    torch.manual_seed(seed)
    lr = lr if lr is not None else DEFAULT_LR[encoder_name]

    model = build_model(
        encoder=encoder_name,
        num_numerical=len(NUMERICAL_COLS),
        cat_cardinalities=cat_cardinalities,
        embedding_dim=embedding_dim,
        kan_grid_size=grid_size,
    ).to(device)

    # Calibracion del grid (hallazgo A3): solo kan-bspline la necesita.
    if hasattr(model, "calibrate"):
        calib_batches = []
        for i, (x_num, _, _) in enumerate(train_loader):
            calib_batches.append(x_num)
            if i >= 4:
                break
        model.calibrate(torch.cat(calib_batches, dim=0).to(device))

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    criterion = torch.nn.BCELoss()

    ckpt_path = f"{CKPT_DIR}/best_{encoder_name}_gs{grid_size}_s{seed}.pt"
    best_val_auc, patience_ctr = 0.0, 0
    t0 = time.time()

    with mlflow.start_run(run_name=f"{encoder_name}-s{seed}"):
        mlflow.log_params({
            "encoder": encoder_name, "seed": seed, "n_train": len(train_ds),
            "embedding_dim": embedding_dim, "grid_size": grid_size, "lr": lr,
            "device": str(device),
        })

        for epoch in range(max_epochs):
            model.train()
            train_loss = 0.0
            for x_num, x_cat, y in train_loader:
                x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
                optimizer.zero_grad()
                y_pred = model(x_num, x_cat).squeeze()
                loss = criterion(y_pred, y)
                if hasattr(model, "entropy_regularization_loss"):
                    loss = loss + model.entropy_regularization_loss()
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            val_auc, val_ll = evaluate(model, val_loader)
            scheduler.step(1 - val_auc)
            mlflow.log_metrics({"train_loss": train_loss, "val_auc": val_auc, "val_logloss": val_ll}, step=epoch)

            if val_auc > best_val_auc:
                best_val_auc, patience_ctr = val_auc, 0
                torch.save(model.state_dict(), ckpt_path)
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    break

        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        test_auc, test_ll = evaluate(model, test_loader)
        elapsed = time.time() - t0
        n_params = sum(p.numel() for p in model.parameters())
        mlflow.log_metrics({"test_auc": test_auc, "test_logloss": test_ll,
                             "train_seconds": elapsed, "n_params": n_params})

    print(f"  {encoder_name:12s} seed={seed:<4d} epochs={epoch+1:<3d} "
          f"val_auc={best_val_auc:.4f} test_auc={test_auc:.4f} test_ll={test_ll:.4f} "
          f"({elapsed:.0f}s)")

    return {"encoder": encoder_name, "seed": seed, "epochs": epoch + 1,
            "val_auc": best_val_auc, "test_auc": test_auc, "test_logloss": test_ll,
            "train_seconds": elapsed, "n_params": n_params, "lr": lr}


In [ ]:
# mlflow>=3.0 dejo el backend de fichero en modo mantenimiento y lo
# bloquea por defecto; sqlite es la via soportada hacia delante.
mlflow.set_tracking_uri(f"sqlite:///{RESULTS_DIR}/mlflow.db")
mlflow.set_experiment("kanrec-model-comparison-colab")

ENCODERS = ["raw", "autodis", "kan-bspline"]
SEEDS = [42, 123, 256]

print(f"Entrenando {len(ENCODERS)} x {len(SEEDS)} = {len(ENCODERS)*len(SEEDS)} modelos...")
print("(mismo backbone, mismos datos, mismas epocas -- la unica variable es el encoder)\n")

results = []
for encoder_name in ENCODERS:
    for seed in SEEDS:
        results.append(train_one_run(encoder_name, seed))


In [ ]:
results_df = pd.DataFrame(results)
results_df["timestamp"] = pd.Timestamp.now().isoformat()
results_df["source"] = "colab-pro-gpu"

summary = results_df.groupby("encoder").agg(
    test_auc_mean=("test_auc", "mean"), test_auc_std=("test_auc", "std"),
    test_logloss_mean=("test_logloss", "mean"), test_logloss_std=("test_logloss", "std"),
    train_seconds_mean=("train_seconds", "mean"),
).round(4)

print("=" * 70)
print("RESUMEN -- media +/- desviacion sobre 3 semillas (42, 123, 256)")
print("=" * 70)
print(summary.to_string())
print("\nNOTA para la memoria: si test_auc_std es del mismo orden que la "
      "diferencia entre dos encoders, esa diferencia NO es significativa. "
      "Dilo asi en vez de quedarte solo con la media (ver seccion 'amenazas "
      "a la validez').")

results_df.to_csv(f"{RESULTS_DIR}/experiment_results_colab.csv", index=False)
summary.to_csv(f"{RESULTS_DIR}/experiment_summary_colab.csv")
print(f"\nGuardado en {RESULTS_DIR}/experiment_results_colab.csv")
print("Sube este CSV tambien a Tables/experiment_results en Fabric si "
      "quieres que Power BI lo consuma junto al resto del proyecto.")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, metric, title in [(axes[0], "test_auc", "AUC (test)"),
                            (axes[1], "test_logloss", "Log-loss (test)")]:
    means = results_df.groupby("encoder")[metric].mean()
    stds  = results_df.groupby("encoder")[metric].std()
    order = ["raw", "autodis", "kan-bspline"]
    ax.bar(order, means[order], yerr=stds[order], capsize=5,
           color=["#9CA3AF", "#F59E0B", "#2563EB"])
    ax.set_title(title)
    ax.set_ylabel(metric)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/comparison_figure.png", dpi=150)
plt.show()
print(f"Figura guardada en {RESULTS_DIR}/comparison_figure.png")


In [ ]:
# Ablacion de grid_size, solo KAN-REC. Barato en Colab con GPU: si grid_size=5
# (el valor por defecto de efficient-kan) es notablemente peor que 10 o 20
# ahora que el grid SI se calibra al rango real de cada campo, es evidencia
# de que la capacidad del spline importa una vez que el rango es correcto.
ablation_results = []
for grid_size in [5, 10, 20]:
    r = train_one_run("kan-bspline", seed=42, grid_size=grid_size, max_epochs=20, patience=2)
    r["grid_size"] = grid_size
    ablation_results.append(r)

ablation_df = pd.DataFrame(ablation_results)
print("\nAblacion de grid_size (KAN-REC, seed=42):")
print(ablation_df[["grid_size", "test_auc", "test_logloss", "train_seconds"]].to_string(index=False))

ablation_df.to_csv(f"{RESULTS_DIR}/gridsize_ablation_colab.csv", index=False)
print(f"\nGuardado en {RESULTS_DIR}/gridsize_ablation_colab.csv")


## Qué hacer con estos resultados

- `experiment_results_colab.csv` y `experiment_summary_colab.csv` en tu
  Drive: son la fuente de la tabla de resultados de la memoria.
- `comparison_figure.png`: figura lista para pegar en la memoria o el vídeo.
- `gridsize_ablation_colab.csv`: resultado adicional barato para la sección
  de resultados (Resultado 2 del criterio del concurso).
- Compara estos números con los de `Tables/experiment_results_trial` en
  Fabric: deberían ser del mismo orden (mismos datos, mismo código), aunque
  no idénticos porque Fabric entrena con menos filas, menos épocas y una
  sola semilla — es la prueba de pipeline, no la cifra final.
